In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from pygraphon.graphons import StepGraphon
from pygraphon.estimators import SmoothNetHist

from utils_generators import AssociativeFullyRandom
from pygraphon.norm import MseProbaEdge, MaeProbaEdge, AUCEdge, AUPRCEdge
import matplotlib.pyplot as plt
from pygraphon.plots.plot_matrices import spy


In [ ]:
import io
from zipfile import ZipFile
import requests
import pandas as pd
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

url =  "https://networks.skewed.de/net/polblogs/files/polblogs.csv.zip"
response = requests.get(url, stream=False)
with ZipFile(io.BytesIO(response.content)) as myzip:
   with myzip.open(myzip.namelist()[0]) as myfile:
      df = pd.read_csv(myfile)
   with myzip.open(myzip.namelist()[1]) as myfile:
      df2 = pd.read_csv(myfile)

df.columns = ["source", "target"]
df2.columns = [s.replace(" ","") for s in df2.columns]
z = df2["value"].values

   
#df2.columns = ["value"] # 0 for liberals, 1 for conservative
G = nx.from_pandas_edgelist(df, "source", "target")
A = nx.to_numpy_array(G, nodelist=sorted(G.nodes()))
# we remove the diagonal to avoid self-loops
A -= np.diag(np.diag(A))
spy(A)

In [ ]:
for index, row in df.iterrows():
    u = row["source"]
    v = row["target"] 
    if not G.has_edge(u, v):
        print("missing edge", u, v)
    if not G.has_edge(v, u):
        print("missing edge", v, u)

In [ ]:
#To check if Sofia is correct also
import networkx as nx
import numpy as np
from pygraphon.plots.plot_matrices import spy

# Load the edge list into a NetworkX graph
G_txt = nx.read_edgelist("polblogs.txt")

# Convert the graph into an adjacency matrix
adjacency_matrix = nx.to_numpy_array(G_txt)

# If you want to represent the adjacency matrix as a NumPy array:
A_olhede = np.array(adjacency_matrix)
A_olhede -= np.diag(np.diag(A_olhede))

In [ ]:
num_liberals = np.sum(z == 0)
num_conservatives = np.sum(z == 1)
print(num_conservatives)
print(num_liberals)

In [ ]:
import networkx as nx
from random import shuffle

# Compute the degrees of the nodes
degrees = dict(G.degree())

def check_cross_citation(graph,node):
    neighbors = graph.neighbors(node)
    for neighbor in neighbors:
        if z[node] != z[neighbor]:
            return True
    return False

# Sort nodes by increasing degrees within each membership group
nodes_0_0 = [node for node in G.nodes() if z[node] == 0 and not check_cross_citation(G,node)]
nodes_0_1 = [node for node in G.nodes() if z[node] == 0 and check_cross_citation(G,node)]
nodes_1_0 = [node for node in G.nodes() if z[node] == 1 and  check_cross_citation(G,node)]
nodes_1_1 = [node for node in G.nodes() if z[node] == 1 and not check_cross_citation(G,node)]

shuffle(nodes_0_0)
shuffle(nodes_1_1)

nodes_0_0_sorted = nodes_0_0 #sorted(nodes_0_0, key=lambda node: degrees[node])
nodes_0_1_sorted = sorted(nodes_0_1, key=lambda node: degrees[node])
nodes_1_0_sorted = sorted(nodes_1_0, key=lambda node: degrees[node], reverse=True)
nodes_1_1_sorted = nodes_1_1 # sorted(nodes_1_1, key=lambda node: degrees[node], reverse=False)

# Create the reordered adjacency matrix
reordered_A = nx.adjacency_matrix(G, nodelist=nodes_0_0_sorted + nodes_0_1_sorted + nodes_1_0_sorted + nodes_1_1_sorted).toarray()

spy(reordered_A, marker_size=1, origin="lower")
plt.show()

# Gradient like polblog

In [ ]:
# Compute the degrees of the nodes
degrees = dict(G.degree())
nodes_0_sorted = sorted(G.nodes(), key=lambda node: degrees[node], reverse=True)
# Create the reordered adjacency matrix
reordered_A = nx.adjacency_matrix(G, nodelist=nodes_0_sorted).toarray()
reordered_A -= np.diag(np.diag(reordered_A))
spy(reordered_A, marker_size=1,  ds="default", origin = "lower")
plt.show()

In [ ]:
est = SmoothNetHist()
est.fit(reordered_A)
first_hist_olhede = est._first_approximate_graphon_from_adjacency(adjacencyMatrix= reordered_A)

In [ ]:
from pygraphon import HistogramEstimator
est_origin = HistogramEstimator()
est_origin.fit(reordered_A)

In [ ]:
print(est.get_ratio_par())
print("Number of unique parameters in SmoothNetHist:", len(np.unique(est.graphon.get_graphon())))
print("Number of unique parameters in Olhede:", len(np.unique(est_origin.get_graphon().graphon)))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(est_origin.get_edge_connectivity(), cmap='jet')#, vmin=0, vmax=1)
im2 = ax[2].imshow(est.get_edge_connectivity(), cmap='jet')#, vmin=0, vmax=1)
_ = spy(reordered_A, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("Olhede histogram")
ax[2].set_title("SmoothNetHist")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)

plt.show()

In [ ]:
plt.imshow(reordered_A,  interpolation = "none")

In [ ]:

fig, ax = plt.subplots(1, 3, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(np.sqrt(est_origin.get_graphon().graphon), cmap='jet')
im2 = ax[2].imshow(np.sqrt(est.get_graphon().graphon), cmap='jet')
_ = spy(reordered_A, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("Olhede histogram")
ax[2].set_title("SmoothNetHist")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)

plt.show()

In [ ]:
import numpy as np
np.diag(np.sqrt(est.get_graphon().graphon))

In [ ]:
graphon_polblogs = est.graphon.get_graphon()
# Get the diagonal elements
diagonal = np.diag(graphon_polblogs)

# Sort the diagonal elements in increasing order and get the sorting indices
sorting_indices = np.argsort(diagonal)

# Permute the rows and columns of the matrix based on the sorting indices
graphon_polblogs_permuted = graphon_polblogs[sorting_indices][:, sorting_indices]

print(graphon_polblogs_permuted.shape)

In [ ]:
fig = plt.figure(figsize=(6, 3.2))

ax = fig.add_subplot(111)
ax.set_title('colorMap')
plt.imshow(graphon_polblogs_permuted, cmap='jet')
ax.set_aspect('equal')

cax = fig.add_axes([0.12, 0.1, 0.78, 0.8])
cax.get_xaxis().set_visible(False)
cax.get_yaxis().set_visible(False)
cax.patch.set_alpha(0)
cax.set_frame_on(False)
plt.colorbar(orientation='vertical')
plt.show()